# Only run this once so the class is defined

In [1]:
#!/usr/bin/env python
# coding: utf-8
import smbus
import time
import math
class YB_Pcb_Car(object):

    def get_i2c_device(self, address, i2c_bus):
        self._addr = address
        if i2c_bus is None:
            return smbus.SMBus(1)
        else:
            return smbus.SMBus(i2c_bus)

    def __init__(self):
        # Create I2C device.
        self._device = self.get_i2c_device(0x16, 1)

    def write_u8(self, reg, data):
        try:
            self._device.write_byte_data(self._addr, reg, data)
        except:
            print ('write_u8 I2C error')

    def write_reg(self, reg):
        try:
            self._device.write_byte(self._addr, reg)
        except:
            print ('write_u8 I2C error')

    def write_array(self, reg, data):
        try:
            # self._device.write_block_data(self._addr, reg, data)
            self._device.write_i2c_block_data(self._addr, reg, data)
        except:
            print ('write_array I2C error')

    def Ctrl_Car(self, l_dir, l_speed, r_dir, r_speed):
        try:
            reg = 0x01
            data = [l_dir, l_speed, r_dir, r_speed]
            self.write_array(reg, data)
        except:
            print ('Ctrl_Car I2C error')
            
    def Control_Car(self, speed1, speed2):
        try:
            if speed1 < 0:
                dir1 = 0
            else:
                dir1 = 1
            if speed2 < 0:
                dir2 = 0
            else:
                dir2 = 1 
            
            self.Ctrl_Car(dir1, int(math.fabs(speed1)), dir2, int(math.fabs(speed2)))
        except:
            print ('Ctrl_Car I2C error')


    def Car_Run(self, speed1, speed2):
        try:
            self.Ctrl_Car(1, speed1, 1, speed2)
        except:
            print ('Car_Run I2C error')

    def Car_Stop(self):
        try:
            reg = 0x02
            self.write_u8(reg, 0x00)
        except:
            print ('Car_Stop I2C error')

    def Car_Back(self, speed1, speed2):
        try:
            self.Ctrl_Car(0, speed1, 0, speed2)
        except:
            print ('Car_Back I2C error')

    def Car_Left(self, speed1, speed2):
        try:
            self.Ctrl_Car(0, speed1, 1, speed2)
        except:
            print ('Car_Spin_Left I2C error')

    def Car_Right(self, speed1, speed2):
        try:
            self.Ctrl_Car(1, speed1, 0, speed2)
        except:
            print ('Car_Spin_Left I2C error')

    def Car_Spin_Left(self, speed1, speed2):
        try:
            self.Ctrl_Car(0, speed1, 1, speed2)
        except:
            print ('Car_Spin_Left I2C error')

    def Car_Spin_Right(self, speed1, speed2):
        try:
            self.Ctrl_Car(1, speed1, 0, speed2)
        except:
            print ('Car_Spin_Right I2C error')

    def Ctrl_Servo(self, id, angle):
        try:
            reg = 0x03
            data = [id, angle]
            if angle < 0:
                angle = 0
            elif angle > 180:
                angle = 180
            self.write_array(reg, data)
        except:
            print ('Ctrl_Servo I2C error') 


# Light & sound demo. 
- Blue light on, buzzer start, buzzer stop, red light on

In [21]:
# Lights & Sound
import RPi.GPIO as GPIO
import time

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BOARD)

# Buzzer
GPIO.setup(32, GPIO.OUT)

led1 = 40
led2 = 38
# LED 1 (Red)
GPIO.setup(led1, GPIO.OUT)
# LED 2 (Blue)
GPIO.setup(led2, GPIO.OUT)

buzzer = None
buzzer = GPIO.PWM(32, 440)

GPIO.output(led2, GPIO.HIGH)
time.sleep(1)
GPIO.output(led2, GPIO.LOW)

buzzer.start(30)
time.sleep(1)
buzzer.stop()

time.sleep(1)
GPIO.output(led1, GPIO.HIGH)
time.sleep(1)
GPIO.output(led1, GPIO.LOW)

GPIO.cleanup()
print('Done.')

Done.


# Turn lights on or off

In [19]:
# Lights & Sound
import RPi.GPIO as GPIO
import time

GPIO.setwarnings(False)
GPIO.setmode(GPIO.BOARD)

# Buzzer
GPIO.setup(32, GPIO.OUT)

led1 = 40
led2 = 38
# LED 1 (Red)
GPIO.setup(led1, GPIO.OUT)
# LED 2 (Blue)
GPIO.setup(led2, GPIO.OUT)

GPIO.output(led2, GPIO.HIGH)

GPIO.output(led1, GPIO.HIGH)


try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    pass

GPIO.output(led2, GPIO.LOW)

GPIO.output(led1, GPIO.LOW)

GPIO.cleanup()
print('Done.')

Done.


# Move car forward

In [22]:
# Movement
import time

car = YB_Pcb_Car()
car.Car_Run(50,50)
time.sleep(3)

car.Car_Stop()
del car

# Car beeps if something is too close ahead of it

In [23]:
# Ultra sonic test
import RPi.GPIO as GPIO
import time

GPIO.setwarnings(False)

EchoPin = 18
TrigPin = 16

#Set GPIO port to BCM coding mode
GPIO.setmode(GPIO.BOARD)

GPIO.setup(EchoPin,GPIO.IN)
GPIO.setup(TrigPin,GPIO.OUT)

# Buzzer
GPIO.setup(32, GPIO.OUT)

buzzer = None
buzzer = GPIO.PWM(32, 440)

#Ultrasonic function
def Distance():
    GPIO.output(TrigPin,GPIO.LOW)
    time.sleep(0.000002)
    GPIO.output(TrigPin,GPIO.HIGH)
    time.sleep(0.000015)
    GPIO.output(TrigPin,GPIO.LOW)

    t3 = time.time()

    while not GPIO.input(EchoPin):
        t4 = time.time()
        if (t4 - t3) > 0.03 :
            return -1
    t1 = time.time()
    while GPIO.input(EchoPin):
        t5 = time.time()
        if(t5 - t1) > 0.03 :
            return -1

    t2 = time.time()
    time.sleep(0.01)
    # print ("distance_1 is %d " % (((t2 - t1)* 340 / 2) * 100))
    return ((t2 - t1)* 340 / 2) * 100

def Distance_test():
    num = 0
    ultrasonic = []
    while num < 5:
            distance = Distance()
            #print("distance is %f"%(distance) )
            while int(distance) == -1 :
                distance = Distance()
                #print("Tdistance is %f"%(distance) )
            while (int(distance) >= 500 or int(distance) == 0) :
                distance = Distance()
                #print("Edistance is %f"%(distance) )
            ultrasonic.append(distance)
            num = num + 1
            time.sleep(0.01)
    distance = (ultrasonic[1] + ultrasonic[2] + ultrasonic[3])/3
    # print("distance is %f"%(distance) ) 
    return distance

try:
    while True:
        distance = Distance_test()
        if distance < 10:
            buzzer.start(30)
            time.sleep(0.5)
        else:
            buzzer.stop()
except KeyboardInterrupt:
    pass
print("Ending") 
GPIO.cleanup()

Ending


# Obstacle avoidance with ultra sonic sensor

In [24]:
# Ultra sonic avoidance
import RPi.GPIO as GPIO
import time

car = YB_Pcb_Car()

car.Car_Run(50, 50)

GPIO.setwarnings(False)

EchoPin = 18
TrigPin = 16

#Set GPIO port to BCM coding mode
GPIO.setmode(GPIO.BOARD)

GPIO.setup(EchoPin,GPIO.IN)
GPIO.setup(TrigPin,GPIO.OUT)

#Ultrasonic function
def Distance():
    GPIO.output(TrigPin,GPIO.LOW)
    time.sleep(0.000002)
    GPIO.output(TrigPin,GPIO.HIGH)
    time.sleep(0.000015)
    GPIO.output(TrigPin,GPIO.LOW)

    t3 = time.time()

    while not GPIO.input(EchoPin):
        t4 = time.time()
        if (t4 - t3) > 0.03 :
            return -1
    t1 = time.time()
    while GPIO.input(EchoPin):
        t5 = time.time()
        if(t5 - t1) > 0.03 :
            return -1

    t2 = time.time()
    time.sleep(0.01)
    # print ("distance_1 is %d " % (((t2 - t1)* 340 / 2) * 100))
    return ((t2 - t1)* 340 / 2) * 100

def Distance_test():
    num = 0
    ultrasonic = []
    while num < 5:
            distance = Distance()
            #print("distance is %f"%(distance) )
            while int(distance) == -1 :
                distance = Distance()
                #print("Tdistance is %f"%(distance) )
            while (int(distance) >= 500 or int(distance) == 0) :
                distance = Distance()
                #print("Edistance is %f"%(distance) )
            ultrasonic.append(distance)
            num = num + 1
            time.sleep(0.01)
    distance = (ultrasonic[1] + ultrasonic[2] + ultrasonic[3])/3
    # print("distance is %f"%(distance) ) 
    return distance

def avoid():
    distance = Distance_test()
    if distance < 30 or distance > 250:
        car.Car_Stop() 
        time.sleep(0.1)
        car.Car_Spin_Right(50,50) 
        time.sleep(0.5)
    else:
        car.Car_Run(50,50) 

try:
    while True:
        avoid()
except KeyboardInterrupt:
    pass
print("Ending")
car.Car_Stop() 
GPIO.cleanup()

Ending
